In [5]:
#####################################################
# DIAGNÓSTICO DE LAS HOJAS PARA EL DASHBOARD DEL SOCIO
# Revisa las 3 hojas del archivo de datos complementarios
#####################################################
#Importamos librerias
import pandas as pd

#####################################################
#DATOS A LLENAR (solo modificar esta sección)
ARCHIVO = "Datos_CLV.xlsx"
HOJAS = ["Existencias_totales", "Histórico_de_Entradas_Salidas", "Histórico_Ventas"]
#####################################################

#####################################################
#Función para describir cualquier hoja
def describir(nombre, df):
    print("\n" + "=" * 60)
    print("HOJA:", nombre)
    print("=" * 60)
    print("Filas y columnas:", df.shape)
    print("\nTipos de dato:")
    print(df.dtypes)
    print("\nNulos por columna (solo las que tienen):")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin nulos")
    print("\nFilas duplicadas completas:", df.duplicated().sum())
    print("\nPrimeras 3 filas:")
    print(df.head(3).to_string())

    #Valores distintos por columna (para ubicar llaves y categorías)
    print("\nValores distintos por columna:")
    print(df.nunique())

    #Columnas de fecha: mostramos mínimo y máximo
    for col in df.columns:
        if "fecha" in col.lower():
            print(f"\nRango de {col}: {df[col].min()}  ->  {df[col].max()}")

    #Columnas de año o mes: mostramos cuántos registros hay de cada valor
    for col in df.columns:
        nombre_col = col.lower()
        if nombre_col in ["anio", "año", "mes"]:
            print(f"\nRegistros por {col}:")
            print(df[col].value_counts(dropna=False).sort_index())

    #Descriptivos de las numéricas
    numericas = df.select_dtypes(['float', 'int'])
    if numericas.shape[1] > 0:
        print("\nDescriptivos numéricos:")
        print(numericas.describe().T.to_string())

#####################################################
#Función para encontrar la columna de artículo y de marca
def buscar_columna(df, palabras):
    for col in df.columns:
        for palabra in palabras:
            if palabra in col.lower():
                return col
    return None

#####################################################
#1. Lectura y descripción de las hojas
#Mostramos las hojas que trae el archivo, por si algún nombre no coincide
print("Hojas que trae el archivo:", pd.ExcelFile(ARCHIVO).sheet_names)
#Leemos solo las hojas indicadas en HOJAS
hojas = pd.read_excel(ARCHIVO, sheet_name=HOJAS)
print("Hojas leídas:", list(hojas.keys()))

for nombre, df in hojas.items():
    describir(nombre, df)

#####################################################
#2. Cruce de llaves entre hojas (artículo y marca)
print("\n" + "=" * 60)
print("CRUCE DE LLAVES ENTRE HOJAS")
print("=" * 60)

articulos = {}
marcas = {}
for nombre, df in hojas.items():
    col_art = buscar_columna(df, ["articulo_id", "articulo"])
    col_marca = buscar_columna(df, ["marca", "marc"])
    print(f"{nombre}: columna artículo = {col_art} | columna marca = {col_marca}")
    if col_art:
        articulos[nombre] = set(df[col_art].dropna().astype(str).str.strip())
    if col_marca:
        marcas[nombre] = set(df[col_marca].dropna().astype(str).str.strip().str.upper())

#Comparamos cada par de hojas
nombres = list(hojas.keys())
for i in range(len(nombres)):
    for j in range(i + 1, len(nombres)):
        a, b = nombres[i], nombres[j]
        if a in articulos and b in articulos:
            comunes = len(articulos[a] & articulos[b])
            print(f"\nArtículos {a} ({len(articulos[a])}) vs {b} ({len(articulos[b])}): {comunes} en común")
        if a in marcas and b in marcas:
            comunes_m = len(marcas[a] & marcas[b])
            print(f"Marcas {a} ({len(marcas[a])}) vs {b} ({len(marcas[b])}): {comunes_m} en común")

#Ejemplos de IDs de cada hoja (para ver si tienen el mismo formato)
print("\nEjemplos de artículo por hoja:")
for nombre, conjunto in articulos.items():
    print(nombre, "->", list(conjunto)[:5])

Hojas que trae el archivo: ['Existencias_totales', 'Histórico_de_Entradas_Salidas', 'Histórico_Ventas']
Hojas leídas: ['Existencias_totales', 'Histórico_de_Entradas_Salidas', 'Histórico_Ventas']

HOJA: Existencias_totales
Filas y columnas: (2951, 6)

Tipos de dato:
articulo_id                    int64
nombre_articulos              object
marca_proveniente             object
almacenes                     object
existencias_id                 int64
valor_inventario_articulo    float64
dtype: object

Nulos por columna (solo las que tienen):
Sin nulos

Filas duplicadas completas: 0

Primeras 3 filas:
   articulo_id                nombre_articulos marca_proveniente         almacenes  existencias_id  valor_inventario_articulo
0         5804     CAMEO PRESS INGOT BOX HT A2             Aidite  Almacén General            1396                  431773.71
1         5816     CAMEO PRESS INGOT BOX LT A1             Aidite  Almacén General            1043                  316841.19
2         5762  CA